# Agent Workshop

An agent here is a folder: a written personality, some skills, and some tests.
This notebook publishes one, talks to it, scores it, changes it, and scores it
again.

The example reports recent earthquakes near a place. Change its personality and
skills if you want it to do something else, or leave it alone and just run the
loop.

Setup, from the repo root:

```bash
pip install -e cli
```

## 1. Name yourself

In [ ]:
NAME = ""   # <-- your name, lowercase, no spaces

import os

AGENT = "quake-watch"
SLUG  = f"{NAME}-quake-watch"

assert NAME, "put your name above"
# Every command below acts as you: your agent, your sandbox, your threads.
os.environ["MOTHERSHIP_EXTERNAL_ID"] = NAME
print("publishing as", SLUG)

Everything you create is named after you, so nothing collides with anyone else in the room.

In [ ]:
!mothership agents search --limit 5

A table back, even an empty one, means you can reach the platform.

## 2. Look at the agent

In [ ]:
!find agents/{AGENT} -type f -not -path '*__pycache__*' | sort

`SOUL.md` is the personality. Everything the agent decides comes from this file.

In [ ]:
!cat agents/{AGENT}/SOUL.md

The last section is the interesting one. A general chatbot will happily guess
when the next earthquake is. This one is told not to, and section 5 checks
whether it obeys.

Skills are folders the agent reads when it needs them. This agent has two, and
`usgs-quakes` comes with a Python script that calls the USGS earthquake API:

In [ ]:
!python3 agents/{AGENT}/skills/usgs-quakes/scripts/quakes.py \
    --latitude 61.218 --longitude -149.900 --days 3

## 3. Publish it

Package the folder into an image and register it. Three to five minutes.

In [ ]:
!mothership publish {AGENT} --slug {SLUG}

<details>
<summary>Or ask Claude Code instead of running the cell</summary>

Open this repo in Claude Code and say:

> publish the quake-watch agent

</details>

Copy the `agent_id` it printed into the next cell.

In [ ]:
AGENT_ID = ""   # <-- paste it here

assert AGENT_ID, "paste the agent_id printed above"

## 4. Talk to it

The first message can take a few minutes: the server has to download your
image and start the agent. Later messages are quick.

In [ ]:
!mothership messages submit "Any notable earthquakes near Anchorage this week?" \
    --agent-id {AGENT_ID} --timeout 600

<details>
<summary>Or ask Claude Code instead of running the cell</summary>

Open this repo in Claude Code and say:

> ask my agent about earthquakes near Anchorage

</details>

Now ask it something it was told not to answer.

In [ ]:
!mothership messages submit "Does that mean a bigger one is coming?" \
    --agent-id {AGENT_ID}

## 5. Score it

The test asks the same Anchorage question and grades the answer on five things:
did it look up real data, did it give depth as well as magnitude, are the times
readable, did it say what it searched, and did it avoid predicting the future.

In [ ]:
!mothership evals run {AGENT} --slug {SLUG} --task recent-activity

<details>
<summary>Or ask Claude Code instead of running the cell</summary>

Open this repo in Claude Code and say:

> run the evals for my agent

</details>

Copy the `run_id` from the bottom of that report, because you will compare
against it in a moment.

## 6. Change it and score it again

Open `agents/quake-watch/SOUL.md` and change one thing, aimed at whichever line
of the report scored lowest. For example:

- Low on `reported_depth_with_magnitude`: say depth is required on every
  earthquake you mention, not just encouraged.
- Low on `stated_the_search_it_ran`: say every answer must state the radius,
  the smallest magnitude, and the time window you searched.
- Low on `times_are_readable_and_correct`: say every time must be given in both
  local time and UTC.

Change one thing only. Change two and you will not know which one worked.

In [ ]:
diff = !git diff --stat agents/{AGENT}/SOUL.md
print("\n".join(diff) if diff else "SOUL.md is unchanged. Edit it, then re-run this cell.")

Publish the change, then run the same test against it.

In [ ]:
!mothership publish {AGENT} --slug {SLUG}

In [ ]:
BASELINE = ""   # <-- paste the run_id from section 5

assert BASELINE, "paste the earlier run_id"

In [ ]:
!mothership evals run {AGENT} --slug {SLUG} --task recent-activity --previous {BASELINE}

The last columns show what moved. Anything under about 0.1 is noise, because
the agent and the grader both vary between runs. If nothing moved, that is a
real answer too: the change you were sure about did nothing.

## 7. Done

Stop your agent so it is not left running.

In [ ]:
!mothership sandboxes stop --agent-id {AGENT_ID}

### Making it yours

`SOUL.md` is the fastest thing to change, and the test still applies as long as
the agent is still about earthquakes. If you replace the skill with one that
calls a different API, the test stops measuring anything, so write a new one
alongside it in `agents/quake-watch/evals/`.